In [4]:
# RAG SGU - PyPDFLoader Toi Gian
print("Flow: config -> ingest -> chunk -> vector store -> query")

Flow: config -> ingest -> chunk -> vector store -> query


In [5]:
!pip install sentence-transformers

In [6]:
import sys
from pathlib import Path

from dotenv import load_dotenv

BASE_DIR = Path.cwd()
SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag_core import RAGConfig, RAGPipeline, configure_runtime_environment

configure_runtime_environment()
load_dotenv(BASE_DIR / ".env")

print(f"Workspace: {BASE_DIR}")
print(f"SRC path: {SRC_DIR}")

d:\HOCTAP\CCNLTHD\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Workspace: d:\HOCTAP\CCNLTHD\sgu-student-rag-main
SRC path: d:\HOCTAP\CCNLTHD\sgu-student-rag-main\src


In [7]:
import os


os.environ["RAG_PDF_DIR"] = str(BASE_DIR / "File_PDFs_OCR")

config = RAGConfig.from_env(base_dir=BASE_DIR)
pipeline = RAGPipeline(config)

print("Configuration loaded:")
print(f"- Raw env LLM_TEMPERATURE: {os.getenv('LLM_TEMPERATURE')}")
print(f"- PDF dir: {config.pdf_dir}")
print(f"- Vector store: {config.vector_store_dir}")
print(f"- Chunk size: {config.chunk_size}")
print(f"- Chunk overlap: {config.chunk_overlap}")
print(f"- Retrieval k: {config.retrieval_k}")
print(f"- Embedding model: {config.embedding_model}")
print(f"- LLM model: {config.llm_model}")
print(f"- LLM temperature: {config.llm_temperature}")
print(f"- LLM max tokens: {config.llm_max_tokens}")

Configuration loaded:
- Raw env LLM_TEMPERATURE: 0.5
- PDF dir: d:\HOCTAP\CCNLTHD\sgu-student-rag-main\File_PDFs_OCR
- Vector store: D:\HOCTAP\CCNLTHD\sgu-student-rag-main\vector_store
- Chunk size: 1600
- Chunk overlap: 200
- Retrieval k: 4
- Embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
- LLM model: gemini-2.5-flash
- LLM temperature: 0.5
- LLM max tokens: 1024


In [8]:
build_result = None

try:
    build_result = pipeline.build_index()
    print("Index built successfully")
    print(f"- Loaded PDFs: {len(build_result.ingestion.loaded_pdf_files)}")
    print(f"- Extracted docs: {len(build_result.ingestion.documents)}")
    print(f"- Created chunks: {len(build_result.chunks)}")

    if build_result.ingestion.scanned_pdf_files:
        print("Warning: Some PDFs have no text-layer (skipped):")
        for file_name in build_result.ingestion.scanned_pdf_files:
            print(f"  - {file_name}")
except ValueError as exc:
    print("Could not build from PDFs:")
    print(exc)
    print("Trying to load existing FAISS index in next cell...")

  0%|          | 0/19 [00:00<?, ?it/s]d:\HOCTAP\CCNLTHD\.venv\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2444.40it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index built successfully
- Loaded PDFs: 19
- Extracted docs: 58
- Created chunks: 98
  - 2022-11-15-2590-QĐ-ĐHSG-QĐ về việc ban hành QUy định quản lí, tổ chức hoạt động khóa luận tốt nghiệp đối với sinh viên trình độ đại học, hệ chính quy.pdf
  - BanMOTa_CNTT_2020-2024.pdf
  - BanMoTaCTDT_CLC_2020-2024.pdf
  - BanMoTa_CTDT_KTPM_2020-2024.pdf
  - CTDT_Thac-si_2019.pdf
  - DeAnTuyenSinh_SGU_2019.pdf
  - QD_DieuChinhKLTN_2025.pdf
  - SGD_DeAnTuyenSinh2018_2018_7_06.pdf
  - TBTS_dot_2-2018-1.pdf
  - TTTN_Kehoachthuctap.pdf


## Build or Load Index


In [9]:
if build_result is None:
    pipeline.load_index()
    print("Loaded existing index from disk")
else:
    print("Using freshly built in-memory index")

def ask(question: str):
    result = pipeline.query(question)
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")

    if result["sources"]:
        print("Sources:")
        for source in result["sources"]:
            print(f"  - {source}")
    else:
        print("Sources: (none)")

    return result

Using freshly built in-memory index


In [10]:
# Override ask() to show detailed retrieved passages in notebook output.
def _source_detail(doc, idx: int) -> str:
    metadata = getattr(doc, "metadata", {}) or {}
    source = (
        metadata.get("source")
        or metadata.get("source_relpath")
        or metadata.get("source_path")
        or metadata.get("file_name")
    )

    page = metadata.get("page_number")
    if page is None and metadata.get("page") is not None:
        try:
            page = int(metadata.get("page")) + 1
        except (TypeError, ValueError):
            page = metadata.get("page")
    elif page is not None:
        try:
            page = int(page)
        except (TypeError, ValueError):
            pass

    label = Path(str(source)).name if source else f"Tài liệu {idx} (thiếu metadata nguồn)"
    if page is not None:
        label = f"{label} - trang {page}"

    content = str(getattr(doc, "page_content", "")).replace("\n", " ").strip()
    preview = content[:180] + ("..." if len(content) > 180 else "")
    return f"{label}: {preview}" if preview else label

def ask(question: str, show_doc_details: bool = True, max_doc_details: int = 5):
    result = pipeline.query(question)
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")

    if result["sources"]:
        print("Sources:")
        for source in result["sources"]:
            print(f"  - {source}")
    else:
        print("Sources: (none)")

    if show_doc_details and result.get("docs"):
        print("\nTop retrieved passages:")
        for idx, doc in enumerate(result["docs"][:max_doc_details], start=1):
            print(f"  {idx}. {_source_detail(doc, idx)}")

    return result

In [11]:
demo_result = ask("Muc tieu dao tao cua nganh Cong nghe thong tin la gi?")

Question: Muc tieu dao tao cua nganh Cong nghe thong tin la gi?
Answer: Tôi không tìm thấy thông tin này trong tài liệu.
Sources:
  - CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 1
  - DRL_Quy-dinh-danh-gia-ket-qua-ren-luyen.pdf - trang 1
  - CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 10
  - CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 8
  - CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 7
  - TBTS_Tien-si_2019-4-nganh.pdf - trang 17
  - TBTS_Tien-si_2019-4-nganh.pdf - trang 11
  - TBTS_Tien-si_2019-4-nganh.pdf - trang 13

Top retrieved passages:
  1. CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 1: ỦY BAN NHÂN DÂN TP. H Ồ CHÍ MINH TR ƯỜNG ĐẠ I H ỌC SÀI GÒN -------------------------- CHƯƠNG TRÌNH ĐÀO TẠO NGÀNH ĐÀO T ẠO : CÔNG NGH Ệ THÔNG TIN TRÌNH ĐỘ ĐÀO T ẠO : ĐẠI H ỌC TP. H ...
  2. DRL_Quy-dinh-danh-gia-ket-qua-ren-luyen.pdf - trang 1: 1 ỦY BAN NHÂN DÂN THÀNH PH Ố HỒ CHÍ MIN H TRƯ ỜNG ĐẠI HỌC S ÀI GÒN C ỘNG H ÒA XÃ H ỘI CHỦ NGHĨA

## Out-of-scope Query Test
Cell tiep theo dung de test cau hoi ngoai tai lieu (ky vong fallback).

In [12]:
out_of_scope_result = ask("Thoi tiet hom nay o TP.HCM nhu the nao?")

Question: Thoi tiet hom nay o TP.HCM nhu the nao?
Answer: Tôi không tìm thấy thông tin này trong tài liệu.
Sources:
  - CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 1
  - CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 10
  - DRL_Quy-dinh-danh-gia-ket-qua-ren-luyen.pdf - trang 1
  - CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 2

Top retrieved passages:
  1. CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 1: ỦY BAN NHÂN DÂN TP. H Ồ CHÍ MINH TR ƯỜNG ĐẠ I H ỌC SÀI GÒN -------------------------- CHƯƠNG TRÌNH ĐÀO TẠO NGÀNH ĐÀO T ẠO : CÔNG NGH Ệ THÔNG TIN TRÌNH ĐỘ ĐÀO T ẠO : ĐẠI H ỌC TP. H ...
  2. CTDT_cong-nghe-thong-tin-he-chuan-2012-2016.pdf - trang 10: 1 ỦY BAN NHÂN DÂN C ỘNG HÒA XÃ H ỘI CH Ủ NGH ĨA VI ỆT NAM TP. H Ồ CHÍ MINH Độc l ập – T ự do – H ạnh phúc TR ƯỜNG ĐẠ I H ỌC SÀI GÒN Thành ph ố H ồ Chí Minh, ngày 14 tháng 8 n ăm 20...
  3. DRL_Quy-dinh-danh-gia-ket-qua-ren-luyen.pdf - trang 1: 1 ỦY BAN NHÂN DÂN THÀNH PH Ố HỒ CHÍ MIN H TRƯ ỜNG ĐẠI HỌC